In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Hilfsfunktion zum Anzeigen von Bildern im Notebook
def show_image(img, title='', cmap=None):
    if len(img.shape) == 2:  # Graustufenbild
        plt.imshow(img, cmap=cmap)
    else:  # Farbbild (BGR zu RGB konvertieren)
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.show()

In [ ]:
bild = cv2.imread('IMP_Foto.jpg')
show_image(bild, title='Eingabebild')

hsv = cv2.cvtColor(bild, cv2.COLOR_BGR2HSV)

h, s, v = cv2.split(hsv)
cv2.imwrite('kanal_H.png', h)
cv2.imwrite('kanal_S.png', s)
cv2.imwrite('kanal_V.png', v)

In [ ]:
gruen_lower = np.array([60, 120, 0])
gruen_upper = np.array([75, 255, 255])

maske = cv2.inRange(hsv, gruen_lower, gruen_upper)

show_image(maske, title='Segmentiertes Grün (Binärbild)', cmap='gray')

In [ ]:
## Morphologische Operationen: Erode und Dilate
kernel = np.ones((5, 5), np.uint8)
maske_erodiert = cv2.erode(maske, kernel, iterations=3)
maske_dilatiert = cv2.dilate(maske_erodiert, kernel, iterations=3)

show_image(maske_dilatiert, title='Bereinigte Maske (Erode + Dilate)', cmap='gray')

In [ ]:
# Inverse Maske berechnen
maske_invers = cv2.bitwise_not(maske_dilatiert)

# Vordergrund (alles außer grün)
vordergrund = cv2.bitwise_and(bild, bild, mask=maske_invers)

# Hintergrundmaske anwenden
hintergrund = cv2.imread('Fussballplatz.jpg')
hintergrund_maskiert = cv2.bitwise_and(hintergrund, hintergrund, mask=maske_dilatiert)

# Vordergrund und neuen Hintergrund kombinieren
res = cv2.add(vordergrund, hintergrund_maskiert)

# Ergebnis anzeigen
show_image(res, title='Green Screen ersetzt')
cv2.imwrite('result.jpg', res)